In [1]:
"""
Project: Customer Lifetime Value and Churn Intelligence System
Tech Stack: Pandas, LightGBM / Scikit-Learn, SHAP, Lifelines (Survival Analysis), Streamlit
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def run_clv_churn_pipeline():
    print("\n=== [DS-81] Running Customer Lifetime Value & Churn System Pipeline ===")
    
    # Step 1: Load/Simulate Telecom/Bank Churn Dataset
    data = {
        'customer_id': [1001, 1002, 1003, 1004, 1005],
        'tenure_months': [12, 2, 24, 1, 36],
        'monthly_charges': [70.5, 95.0, 55.2, 105.0, 45.0],
        'total_charges': [846.0, 190.0, 1324.8, 105.0, 1620.0],
        'complaints_filed': [0, 3, 1, 5, 0],
        'churn': [0, 1, 0, 1, 0] # 1 = Churned, 0 = Retained
    }
    df = pd.DataFrame(data)
    
    # Step 2: Feature Engineering & CLV Calculation
    # CLV = (Monthly Charges * Tenure) adjusted for retention
    df['estimated_clv'] = df['monthly_charges'] * df['tenure_months'] * 1.2
    
    # Step 3: Churn Classifier Model (Scikit-Learn / LightGBM simulation)
    X = df[['tenure_months', 'monthly_charges', 'total_charges', 'complaints_filed']]
    y = df['churn']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    
    df['churn_probability'] = model.predict_proba(X)[:, 1]
    
    # Step 4: Retention Offer Recommendation Engine & Uplift logic
    def assign_retention_offer(row):
        if row['churn_probability'] > 0.5 and row['estimated_clv'] > 1000:
            return "High-Value Retention: Offer 20% Discount + Free Upgrade"
        elif row['churn_probability'] > 0.5:
            return "Standard Retention: Send 10% Coupon"
        else:
            return "No Offer Needed (Loyal Customer)"
            
    df['recommended_action'] = df.apply(assign_retention_offer, axis=1)
    
    print("\nProcessed CLV & Churn Intelligence DataFrame:")
    print(df[['customer_id', 'tenure_months', 'estimated_clv', 'churn_probability', 'recommended_action']])
    return df

if __name__ == "__main__":
    run_clv_churn_pipeline()


=== [DS-81] Running Customer Lifetime Value & Churn System Pipeline ===

Processed CLV & Churn Intelligence DataFrame:
   customer_id  tenure_months  estimated_clv  churn_probability  \
0         1001             12        1015.20               0.09   
1         1002              2         228.00               0.62   
2         1003             24        1589.76               0.01   
3         1004              1         126.00               0.71   
4         1005             36        1944.00               0.01   

                    recommended_action  
0     No Offer Needed (Loyal Customer)  
1  Standard Retention: Send 10% Coupon  
2     No Offer Needed (Loyal Customer)  
3  Standard Retention: Send 10% Coupon  
4     No Offer Needed (Loyal Customer)  
